## With manual log-in but automatic Chrome opening

In [33]:
import requests
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import time, re, json, os
import pandas as pd
from tabulate import tabulate
from datetime import datetime

# ---------- EXPERIENCE PARSER CLASS ----------
class ExperienceParser:
    def __init__(self, driver):
        self.driver = driver

    def clean_text(self, txt):
        return re.sub(r"(See more|Mostra|Vedi).*", "", txt or "", flags=re.I).strip()

    def parse_from_main_profile(self):
        """Parse experiences from the main profile page"""
        experiences = []
        try:
            exp_section = self.driver.find_element(
                By.XPATH,
                "//section[contains(@id,'experience') or .//h2[contains(.,'Experience')]]"
            )
            exp_items = exp_section.find_elements(By.XPATH, ".//li[contains(@class,'artdeco-list__item')]")

            for item in exp_items:
                try:
                    role_check = item.find_element(By.XPATH, ".//div[contains(@class,'t-bold')]/span").text.strip()
                    if not role_check:
                        continue
                except:
                    continue

                # Get company name
                try:
                    company = item.find_element(
                        By.XPATH, ".//div[contains(@class,'t-bold')]/span"
                    ).text.strip()
                except:
                    company = ""

                # --- detect nested roles ---
                subroles = item.find_elements(By.XPATH, ".//ul/li")
                valid_subroles = []
                for sub in subroles:
                    try:
                        sub.find_element(By.XPATH, ".//div[contains(@class,'t-bold')]/span").text.strip()
                        valid_subroles.append(sub)
                    except:
                        continue

                if valid_subroles:
                    for sub in valid_subroles:
                        exp = self.extract_role(sub, parent_company=company)
                        experiences.append(exp)
                else:
                    exp = self.extract_role(item)
                    experiences.append(exp)

        except Exception as e:
            print("⚠️ Experience parsing error:", e)

        return experiences

    def parse_from_detail_page(self):
        """Parse experiences from the detailed experience page"""
        experiences = []
        try:
            # Wait for the page to load
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "main"))
            )
            
            # Additional wait for content to load
            time.sleep(2)
            
            # Find all experience items
            exp_items = self.driver.find_elements(
                By.XPATH, 
                "//ul[@class='TQtryBgeHzFGDGLkqnUeDFeVWQtCnNDoJbgs']/li[contains(@class,'pvs-list__paged-list-item')]"
            )
            
            print(f"📊 Found {len(exp_items)} experience items on detail page")

            for item in exp_items:
                try:
                    # Check if this is a company grouping (multiple roles at same company)
                    company_group = item.find_elements(
                        By.XPATH, 
                        ".//div[@class='pvs-list__container']//ul[@class='TQtryBgeHzFGDGLkqnUeDFeVWQtCnNDoJbgs']"
                    )
                    
                    if company_group:
                        # This is a company with multiple roles
                        company_name = self.extract_company_name(item)
                        
                        # Find all sub-roles
                        subroles = company_group[0].find_elements(
                            By.XPATH, 
                            "./li[contains(@class,'pvs-list__paged-list-item')]"
                        )
                        
                        for subrole in subroles:
                            exp = self.extract_role_from_detail(subrole, parent_company=company_name)
                            if exp and exp.get('job_title'):
                                experiences.append(exp)
                    else:
                        # Single role at company
                        exp = self.extract_role_from_detail(item)
                        if exp and exp.get('job_title'):
                            experiences.append(exp)
                            
                except Exception as e:
                    print(f"⚠️ Error parsing experience item: {e}")
                    continue

        except Exception as e:
            print(f"⚠️ Error parsing detail page: {e}")

        return experiences

    def extract_company_name(self, node):
        """Extract company name from a grouped experience node"""
        try:
            company_elem = node.find_element(
                By.XPATH,
                ".//div[contains(@class,'t-bold')]/span[@aria-hidden='true']"
            )
            return self.clean_text(company_elem.text)
        except:
            return ""

    def extract_role_from_detail(self, node, parent_company=""):
        """Extract role information from detail page structure"""
        try:
            # Job title
            role = node.find_element(
                By.XPATH,
                ".//div[contains(@class,'t-bold')]/span[@aria-hidden='true']"
            ).text.strip()
        except:
            role = ""

        # Company name (if not from parent)
        if not parent_company:
            try:
                company_elem = node.find_element(
                    By.XPATH,
                    ".//span[@class='t-14 t-normal']/span[@aria-hidden='true']"
                )
                company_text = company_elem.text.strip()
                # Remove employment type (Full-time, Part-time, etc.)
                company = re.sub(r'\s*·\s*(Full-time|Part-time|Internship|Contract|Freelance).*', '', company_text)
            except:
                company = ""
        else:
            company = parent_company

        # Duration
        duration = ""
        try:
            duration_elem = node.find_element(
                By.XPATH,
                ".//span[contains(@class,'t-black--light')]/span[@class='pvs-entity__caption-wrapper']"
            )
            duration = duration_elem.text.strip()
        except:
            pass

        start_date, end_date, still_work_here = self.parse_duration(duration)

        return {
            "job_title": self.clean_text(role),
            "company": self.clean_text(company),
            "start_date": start_date,
            "end_date": end_date,
            "still_work_here": still_work_here
        }

    def extract_role(self, node, parent_company=""):
        """Extract role information from main profile page (legacy method)"""
        try:
            role = node.find_element(By.XPATH,
                ".//div[contains(@class,'t-bold')]/span").text.strip()
        except:
            role = ""

        try:
            company = node.find_element(By.XPATH,
                ".//span[contains(@class,'t-14') and not(contains(@class,'t-black--light'))]/span").text.strip()
        except:
            company = parent_company

        duration = ""
        light_spans = node.find_elements(By.XPATH,
            ".//span[contains(@class,'t-14') and contains(@class,'t-black--light')]/span")
        for span in light_spans:
            txt = span.text.strip()
            if re.search(r"\d{4}", txt):
                duration = txt
                break

        start_date, end_date, still_work_here = self.parse_duration(duration)

        return {
            "job_title": self.clean_text(role),
            "company": self.clean_text(company),
            "start_date": start_date,
            "end_date": end_date,
            "still_work_here": still_work_here
        }

    def parse_duration(self, duration_text):
        start_date, end_date, still_work_here = "", "", False
        if not duration_text:
            return start_date, end_date, still_work_here

        m = re.search(r"([A-Za-zÀ-ÿ]{3,9} \d{4})\s*[-–]\s*([A-Za-zÀ-ÿ]{3,9} \d{4}|Present|Oggi|Aujourd'hui|Actualidad)?", duration_text, re.I)
        if m:
            start_date = m.group(1)
            end_date = m.group(2) if len(m.groups()) > 1 else ""
            if end_date and re.search(r"present|oggi|current|actualidad|maintenant", end_date, re.I):
                still_work_here = True
                end_date = ""
        else:
            years = re.findall(r"\d{4}", duration_text)
            if years:
                start_date = years[0]
                if len(years) > 1:
                    end_date = years[1]

        return start_date, end_date, still_work_here

# ---------- EDUCATION PARSER CLASS ----------
class EducationParser:
    def __init__(self, driver):
        self.driver = driver

    def clean_text(self, txt):
        txt = re.sub(r"(See more|Mostra|Vedi).*", "", txt or "", flags=re.I).strip()
        return txt.split("\n")[0].strip()

    def parse(self):
        education_list = []
        try:
            edu_section = self.driver.find_element(
                By.XPATH, "//section[@id='education' or .//h2[contains(.,'Education')]]"
            )
            edu_items = edu_section.find_elements(By.XPATH, ".//li[contains(@class,'artdeco-list__item')]")

            for item in edu_items:
                try:
                    institution = self.clean_text(item.find_element(
                        By.XPATH, ".//div[contains(@class,'t-bold')]/span"
                    ).text)
                except:
                    institution = ""

                try:
                    full_text = self.clean_text(
                        item.find_element(By.XPATH, ".//span[@class='t-14 t-normal']").text
                    )
                    if "," in full_text:
                        qualification_type, subject = [x.strip() for x in full_text.split(",", 1)]
                    else:
                        qualification_type = full_text
                        subject = ""
                except:
                    qualification_type, subject = "", ""

                start_date, end_date, still_studying = "", "", False
                try:
                    dates_text = self.clean_text(item.find_element(
                        By.XPATH, ".//span[contains(@class,'t-14') and contains(@class,'t-normal') and contains(@class,'t-black--light')]"
                    ).text)
                    start_date, end_date, still_studying = self.parse_duration(dates_text)
                except:
                    pass

                education_list.append({
                    "institution": institution,
                    "qualification_type": qualification_type,
                    "subject": subject,
                    "start_date": start_date,
                    "end_date": end_date,
                    "still_studying": still_studying
                })

            def get_start_year(edu):
                if edu["start_date"]:
                    m = re.search(r"\d{4}", edu["start_date"])
                    return int(m.group(0)) if m else 0
                return 0

            education_list.sort(key=get_start_year, reverse=True)

        except Exception as e:
            print("⚠️ Education parsing error:", e)

        return education_list

    def parse_duration(self, duration_text):
        start_date, end_date, still_studying = "", "", False
        if not duration_text:
            return start_date, end_date, still_studying

        m = re.search(
            r"([A-Za-zÀ-ÿ]{3,9} \d{4}|\d{4})\s*[-–]\s*([A-Za-zÀ-ÿ]{3,9} \d{4}|\d{4}|Present|Oggi|Aujourd'hui|Actualidad)?",
            duration_text,
            re.I
        )
        if m:
            start_date = m.group(1)
            end_date = m.group(2) if len(m.groups()) > 1 else ""
            if end_date and re.search(r"present|oggi|current|actualidad|maintenant", end_date, re.I):
                still_studying = True
                end_date = ""
        else:
            years = re.findall(r"\d{4}", duration_text)
            if years:
                start_date = years[0]
                if len(years) > 1:
                    end_date = years[1]
        return start_date, end_date, still_studying

# ---------- HELPER FUNCTIONS ----------
def safe_click_element(driver, element, method="javascript"):
    """
    Safely click an element using multiple fallback methods
    """
    try:
        if method == "javascript":
            driver.execute_script("arguments[0].click();", element)
        elif method == "scroll_and_click":
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
            time.sleep(0.5)
            element.click()
        elif method == "action_chains":
            from selenium.webdriver.common.action_chains import ActionChains
            actions = ActionChains(driver)
            actions.move_to_element(element).click().perform()
        return True
    except Exception as e:
        print(f"⚠️ Click method '{method}' failed: {e}")
        return False

def click_show_all_experiences(driver):
    """
    Find and click the 'Show all experiences' button with multiple fallback strategies
    """
    selectors = [
        "//a[@id='navigation-index-see-all-experiences']",
        "//a[contains(@href,'/details/experience')]",
        "//a[contains(.,'Show all') and contains(.,'experience')]",
        "//div[contains(@class,'pvs-list__footer-wrapper')]//a[contains(@href,'experience')]"
    ]
    
    for selector in selectors:
        try:
            element = driver.find_element(By.XPATH, selector)
            print(f"✅ Found 'Show all experiences' button")
            
            # Try multiple click methods
            for method in ["javascript", "scroll_and_click", "action_chains"]:
                if safe_click_element(driver, element, method):
                    print(f"✅ Clicked using {method}")
                    return True
                time.sleep(0.5)
                
        except Exception as e:
            continue
    
    return False

# ---------- CONFIG ----------
PROFILE_URL = "https://www.linkedin.com/in/eliana-di-lodovico-570171192/en"
CHROMEDRIVER_PATH = r"C:\\Users\\Utente\\Downloads\\chromedriver-win64\\chromedriver-win64\\chromedriver.exe"
WAIT_TIMEOUT = 300

# ---------- LAUNCH BROWSER ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
driver = uc.Chrome(driver_executable_path=CHROMEDRIVER_PATH, options=options)
driver.get("https://www.linkedin.com/login")
print("Please log in manually in the browser window...")

start = time.time()
while True:
    time.sleep(1)
    if any(c.get("name") == "li_at" for c in driver.get_cookies()):
        print("✅ Login detected — proceeding.")
        break
    if time.time() - start > WAIT_TIMEOUT:
        input("⚠️ Timeout reached. Press Enter if already logged in.")
        break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(PROFILE_URL)
time.sleep(2)

# ---------- UTILITIES ----------
def safe_click(xpath):
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].scrollIntoView(true);", el)
        time.sleep(0.3)
        el.click()
        time.sleep(0.7)
        return True
    except:
        return False

# Expand sections on main profile
expanders = [
    "//button[contains(.,'See more')]",
    "//button[contains(.,'Mostra')]",
    "//button[contains(@aria-label,'See more')]"
]
for xp in expanders:
    safe_click(xp)

# Scroll to load all content
for f in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{f});")
    time.sleep(0.7)

# ---------- BASIC INFO ----------
def try_selectors(selectors):
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except:
            pass
    return None

name = try_selectors([(By.CSS_SELECTOR, "h1.text-heading-xlarge"), (By.XPATH, "//main//h1")])
headline = try_selectors([(By.CSS_SELECTOR, "div.text-body-medium.break-words")])
location = try_selectors([(By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words")])
about = try_selectors([(By.XPATH, "//section[contains(@class,'pv-about-section')]//p")])

# ---------- PROFILE PICTURE ----------
profile_pic = ""
try:
    img_el = driver.find_element(By.XPATH,
        "//img[contains(@class,'pv-top-card-profile-picture__image') or contains(@class,'profile-photo-edit__preview')]")
    img_url = img_el.get_attribute("src") or ""

    if img_url:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        img_filename = f"profile_pic_{ts}.jpeg"

        try:
            response = requests.get(img_url, stream=True, timeout=10)
            if response.status_code == 200:
                with open(img_filename, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
                profile_pic = img_filename
                print(f"🖼️ Profile picture saved as {img_filename}")
            else:
                print("⚠️ Could not download profile picture, bad status:", response.status_code)
        except Exception as e:
            print("⚠️ Error downloading profile picture:", e)
except:
    profile_pic = ""

# ---------- CONTACT INFO / EMAIL ----------
email = ""
try:
    if safe_click("//a[contains(@href,'overlay/contact-info')]"):
        time.sleep(2)
        email_el = driver.find_element(By.XPATH, "//a[starts-with(@href,'mailto:')]")
        email = email_el.get_attribute("href").replace("mailto:", "").strip()
        safe_click("//button[@aria-label='Dismiss']")
except:
    email = ""

# ---------- EXPERIENCE (ENHANCED) ----------
print("\n🔍 Attempting to scrape experiences from detail page...")

parser = ExperienceParser(driver)
experience = []

try:
    # Scroll to experience section first
    try:
        exp_section = driver.find_element(
            By.XPATH,
            "//section[contains(@id,'experience') or .//h2[contains(.,'Experience')]]"
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", exp_section)
        time.sleep(1)
    except:
        pass
    
    # Try to click "Show all experiences" button
    if click_show_all_experiences(driver):
        print("✅ Navigated to experience detail page")
        time.sleep(3)
        
        # Scroll the detail page to load all content
        for f in [0.25, 0.5, 0.75, 1.0]:
            driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{f});")
            time.sleep(0.5)
        
        # Parse from detail page
        experience = parser.parse_from_detail_page()
        
        if experience:
            print(f"✅ Scraped {len(experience)} experiences from detail page")
        else:
            print("⚠️ No experiences found on detail page, falling back to main profile...")
            driver.get(PROFILE_URL)
            time.sleep(2)
            experience = parser.parse_from_main_profile()
    else:
        raise Exception("Could not click 'Show all experiences' button")
    
except Exception as e:
    print(f"⚠️ Could not access experience detail page: {e}")
    print("📋 Falling back to main profile parsing...")
    
    # Navigate back to main profile
    driver.get(PROFILE_URL)
    time.sleep(2)
    
    # Parse from main profile
    experience = parser.parse_from_main_profile()
    print(f"✅ Scraped {len(experience)} experiences from main profile")

work_experience = pd.DataFrame(experience)

# ---------- EDUCATION ----------
# Navigate back to main profile for education
driver.get(PROFILE_URL)
time.sleep(2)

edu_parser = EducationParser(driver)
education = edu_parser.parse()
education_history = pd.DataFrame(education)

# ---------- DISPLAY ----------
print("\n=== LINKEDIN PROFILE ===")
print(f"👤 Name: {name}")
print(f"💼 Headline: {headline}")
print(f"📍 Location: {location}")
if about:
    print(f"📝 About: {about}")
if profile_pic:
    print(f"🖼️ Profile picture: {profile_pic}")
if email:
    print(f"📧 Email: {email}")

if not work_experience.empty:
    print("\n=== WORK EXPERIENCE ===")
    print(tabulate(work_experience, headers='keys', tablefmt='fancy_grid', showindex=False))

if not education_history.empty:
    print("\n=== EDUCATION HISTORY ===")
    print(tabulate(education_history, headers='keys', tablefmt='fancy_grid', showindex=False))

# ---------- CREATE PROFILE DATAFRAME ----------
first_name, surname = "", ""
if name:
    parts = name.split()
    if len(parts) >= 2:
        first_name = parts[0]
        surname = " ".join(parts[1:])
    else:
        first_name = name

bio = about or headline or ""
cv_url = ""
user_type = ""

profile = pd.DataFrame([{
    "first_name": first_name,
    "surname": surname,
    "email": email,
    "profile_pic": profile_pic,
    "cv_url": cv_url,
    "bio": bio,
    "user_type": user_type
}])

# ---------- SAVE JSON & CSV ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
json_file = f"linkedin_profile_{ts}.json"
csv_exp = f"work_experience_{ts}.csv"
csv_edu = f"education_history_{ts}.csv"
csv_basic = f"profile_{ts}.csv"

with open(json_file, "w", encoding="utf-8") as f:
    json.dump({
        "first_name": first_name,
        "surname": surname,
        "email": email,
        "profile_pic": profile_pic,
        "cv_url": cv_url,
        "bio": bio,
        "user_type": user_type,
        "work_experience": work_experience.to_dict(orient="records"),
        "education_history": education_history.to_dict(orient="records")
    }, f, ensure_ascii=False, indent=2)

work_experience.to_csv(csv_exp, index=False)
education_history.to_csv(csv_edu, index=False)
profile.to_csv(csv_basic, index=False)

print(f"\n✅ JSON saved to {json_file}")
print(f"✅ Work experience CSV saved to {csv_exp}")
print(f"✅ Education history CSV saved to {csv_edu}")
print(f"✅ Profile CSV saved to {csv_basic}")

driver.quit()

Please log in manually in the browser window...
✅ Login detected — proceeding.
🖼️ Profile picture saved as profile_pic_20251107_125329.jpeg

🔍 Attempting to scrape experiences from detail page...
✅ Found 'Show all experiences' button
✅ Clicked using javascript
✅ Navigated to experience detail page
📊 Found 0 experience items on detail page
⚠️ No experiences found on detail page, falling back to main profile...

=== LINKEDIN PROFILE ===
👤 Name: Eliana Di Lodovico
💼 Headline: PhD student in soil science
📍 Location: Pirmasens, Rhineland-Palatinate, Germany
🖼️ Profile picture: profile_pic_20251107_125329.jpeg

=== WORK EXPERIENCE ===
╒═════════════════════════════════╤═════════════════════════════════════════════╤══════════════╤════════════╤═══════════════════╕
│ job_title                       │ company                                     │ start_date   │ end_date   │ still_work_here   │
╞═════════════════════════════════╪═════════════════════════════════════════════╪══════════════╪═══════

In [35]:
work_experience

,job_title,company,start_date,end_date,still_work_here
0,Data Engineer intern,Wdoit · Internship,Sep 2025,,True
1,Researcher Assistant,RPTU Kaiserslautern-Landau,Feb 2022,,True
2,PhD student,Helmholtz Centre for Environmental Research,Feb 2022,Jan 2025,False
3,Junior Environmental Consultant,Progetto per l'Ambiente,Oct 2021,Dec 2021,False
4,Stage,Progetto per l'Ambiente,Jun 2021,Sep 2021,False
5,Research Trainee,INRAE,Jul 2019,Sep 2019,False


In [36]:
education_history

,institution,qualification_type,subject,start_date,end_date,still_studying
0,CodeOp,Bootcamp,Data Science,Jan 2025,Aug 2025,False
1,UNINFORM GROUP,Master II level,"MASTER QUALITY, EXPERTS AND MANAGERS IN INTEGR...",2021,2021,False


In [37]:
profile

,first_name,surname,email,profile_pic,cv_url,bio,user_type
0,Eliana,Di Lodovico,,profile_pic_20251107_125329.jpeg,,PhD student in soil science,
